# Mini-Project — MCP + Agents AI integration with **Gemini**

**Theme: Dev Assistant** 🛠️

An end-to-end *agentic* application where **Gemini** orchestrates **three MCP servers**
through a flexible, tool-driven policy (the LLM decides the next step — nothing is hard-coded).

| # | MCP Server | Type | Transport | What it gives the agent |
|---|-----------|------|-----------|-------------------------|
| 1 | `@modelcontextprotocol/server-filesystem` | third-party (Node) | stdio | list / read / write files in the workspace |
| 2 | `mcp-server-git` | third-party (Python) | stdio | `git status`, `git log`, `git diff`, `git show` … |
| 3 | `dev_ops` (**custom**, FastMCP) | our own (Python) | stdio | `code_metrics`, `generate_changelog`, `suggest_commit_message`, `ping` |

**Composition demo:** the agent reads a file (server 1), inspects the git diff (server 2),
then feeds that text into our custom analysis tools (server 3) — all decided by Gemini at runtime.

> **Environment:** built for **Google Colab**. It also runs locally (falls back to the current
> working directory instead of `/content`).

## How it works

```
                    ┌───────────────────────────┐
   your query  ───► │   Gemini (ReAct agent)     │  ← LLM-driven tool policy
                    │   langgraph create_react   │
                    └─────────────┬─────────────┘
                                  │  picks tools dynamically
              ┌───────────────────┼────────────────────┐
              ▼                   ▼                     ▼
      ┌──────────────┐   ┌────────────────┐   ┌──────────────────┐
      │  filesystem  │   │      git       │   │  dev_ops (custom)│
      │  (npx, Node) │   │ (python -m …)  │   │  (FastMCP, py)   │
      └──────────────┘   └────────────────┘   └──────────────────┘
                    all connected over stdio via
                MultiServerMCPClient (langchain-mcp-adapters)
```

## 1 · Colab setup — install dependencies

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio"

We also need **FastMCP** (to write our own server) and **mcp-server-git** (the third-party git server).

In [ ]:
%pip install -qU "fastmcp>=2.0.0" "mcp-server-git"

## 2 · Set `GOOGLE_API_KEY`

Get a key from [Google AI Studio](https://aistudio.google.com/apikey).
In Colab, the recommended way is the **🔑 Secrets** panel (left sidebar): add a secret named
`GOOGLE_API_KEY` and enable notebook access. The cell below reads it from there, and falls back
to a hidden prompt if the secret isn't set.

In [ ]:
import os

# 1) Colab Secrets  →  2) already-set env var  →  3) hidden prompt
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        import getpass
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your GOOGLE_API_KEY: ")

print("GOOGLE_API_KEY configured:", bool(os.environ.get("GOOGLE_API_KEY")))

## 3 · Confirm Node / NPM (needed for the filesystem server)

The filesystem server ships as a Node package run via `npx`. Colab already has Node pre-installed.

In [ ]:
!node --version || echo "node NOT found"
!npx --version  || echo "npx NOT found"

In [ ]:
# Run this ONLY if node/npx were missing above (uncomment):
# !apt-get -qq update
# !apt-get -qq install -y nodejs npm
# !node --version
# !npx --version

## 4 · Prepare a workspace + git repo

MCP servers act on a real directory, so we create a small sample project and initialise a git
repo with two commits **and one uncommitted change** — that way `git log`, `git diff` and our
changelog tools all have something real to work on.

In [ ]:
import os, subprocess, textwrap
from pathlib import Path

BASE = "/content" if os.path.isdir("/content") else os.getcwd()
WORKDIR = os.path.join(BASE, "dev_workspace")
os.makedirs(WORKDIR, exist_ok=True)

# ---- sample source files -------------------------------------------------
Path(WORKDIR, "app.py").write_text(textwrap.dedent("""
    \"\"\"Tiny demo app.\"\"\"

    def add(a, b):
        return a + b

    def main():
        # TODO: read numbers from CLI args
        print(add(2, 3))

    if __name__ == "__main__":
        main()
"""))

Path(WORKDIR, "utils.py").write_text(textwrap.dedent("""
    def slugify(text):
        # FIXME: handle unicode properly
        return text.strip().lower().replace(" ", "-")
"""))

Path(WORKDIR, "README.md").write_text("# Dev Workspace\n\nSample project used by the MCP Dev Assistant.\n")

# ---- git repo with history ----------------------------------------------
def sh(cmd):
    r = subprocess.run(cmd, cwd=WORKDIR, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

sh("git init -q")
sh("git config user.email 'dev@example.com'")
sh("git config user.name 'Dev Assistant'")
sh("git add -A && git commit -q -m 'chore: initial project scaffold'")

# second commit: add a greeting helper
with open(os.path.join(WORKDIR, "app.py"), "a") as f:
    f.write("\n\ndef greet(name):\n    return f'Hello, {name}!'\n")
sh("git add -A && git commit -q -m 'feat: add greet() helper'")

# uncommitted change so `git diff` shows something
with open(os.path.join(WORKDIR, "utils.py"), "a") as f:
    f.write("\n\ndef shout(text):\n    # TODO: strip punctuation\n    return text.upper()\n")

print("WORKDIR:", WORKDIR)
print("Files  :", os.listdir(WORKDIR))
print("\n--- git log ---")
print(sh("git log --oneline"))
print("\n--- git status ---")
print(sh("git status -s"))

## 5 · Implement the custom MCP server (`dev_ops`) with FastMCP

We write a small stdio server exposing four dev-assistant tools. The **docstrings matter** —
they become the tool descriptions Gemini reads when deciding what to call.

We use the `%%writefile` magic so the file content has no string-escaping issues.

In [ ]:
%%writefile custom_mcp_server.py
from fastmcp import FastMCP
from typing import Dict, List
import re

mcp = FastMCP(name="dev_ops")


@mcp.tool
def ping() -> str:
    """Health check. Returns 'pong' if the custom dev_ops server is alive."""
    return "pong"


@mcp.tool
def code_metrics(code: str) -> Dict[str, int]:
    """Compute basic metrics for a source-code string: total lines, non-empty lines,
    number of Python function definitions, number of class definitions, and number of
    TODO/FIXME markers. Call this AFTER reading a file's contents to summarize its quality."""
    lines = code.splitlines()
    return {
        "total_lines": len(lines),
        "nonempty_lines": sum(1 for l in lines if l.strip()),
        "functions": len(re.findall(r"^\s*def\s+\w+", code, re.MULTILINE)),
        "classes": len(re.findall(r"^\s*class\s+\w+", code, re.MULTILINE)),
        "todos": len(re.findall(r"\b(?:TODO|FIXME)\b", code)),
    }


@mcp.tool
def generate_changelog(diff: str) -> Dict[str, List[str]]:
    """Turn a unified git diff into a categorized changelog. Returns the list of changed files
    plus the added and removed lines. Pass the raw output of `git diff` here."""
    added, removed, files = [], [], []
    for line in diff.splitlines():
        if line.startswith("diff --git"):
            parts = line.split()
            if len(parts) >= 3:
                files.append(parts[2].replace("a/", "", 1))
        elif line.startswith("+++") or line.startswith("---"):
            continue
        elif line.startswith("+"):
            added.append(line[1:].strip())
        elif line.startswith("-"):
            removed.append(line[1:].strip())
    return {
        "files_changed": files,
        "added": [a for a in added if a][:50],
        "removed": [r for r in removed if r][:50],
    }


@mcp.tool
def suggest_commit_message(diff: str) -> str:
    """Heuristically suggest a Conventional-Commits style message ('feat:', 'fix:', 'docs:',
    'test:') from a unified git diff."""
    lower = diff.lower()
    if "def test" in lower or "/tests/" in lower:
        kind = "test"
    elif "readme" in lower or ".md" in lower:
        kind = "docs"
    elif "fix" in lower or "bug" in lower:
        kind = "fix"
    else:
        kind = "feat"
    files = [l.split()[-1].replace("b/", "", 1)
             for l in diff.splitlines() if l.startswith("diff --git")]
    scope = files[0].split("/")[-1] if files else "workspace"
    return f"{kind}: update {scope} ({len(files)} file(s) changed)"


if __name__ == "__main__":
    mcp.run(transport="stdio")

## 6 · Connect to all three MCP servers

`MultiServerMCPClient` spawns each server as a stdio subprocess and exposes their tools as
LangChain tools. We enable `nest_asyncio` so async calls work inside the notebook.

In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()
from langchain_mcp_adapters.client import MultiServerMCPClient

SERVER_PATH = os.path.abspath("custom_mcp_server.py")  # %%writefile wrote it into cwd

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    "dev_ops": {                       # our custom FastMCP server
        "transport": "stdio",
        "command": "python",
        "args": [SERVER_PATH],
    },
}

# tool_name_prefix keeps names unique per server; guard in case the arg isn't supported.
try:
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
except TypeError:
    client = MultiServerMCPClient(mcp_connections)

def run(coro):
    """Run an async coroutine from a notebook cell."""
    return asyncio.get_event_loop().run_until_complete(coro)

tools = run(client.get_tools())

print(f"Connected. Total tools available: {len(tools)}\n")
for t in tools:
    print(f"  - {t.name}")

## 7 · Build the Gemini agent

We use `create_react_agent` (LangGraph). It gives Gemini **all** the tools and lets the model
decide, step by step, which to call — a genuine tool-driven policy, not a scripted flow.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

MODEL = "gemini-2.0-flash"   # also works: "gemini-2.5-flash", "gemini-1.5-pro"

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

SYSTEM_PROMPT = (
    "You are a Dev Assistant with access to three MCP toolsets: a filesystem, git, and a custom "
    "dev_ops server (code_metrics, generate_changelog, suggest_commit_message, ping). "
    "Decide which tools to call to satisfy the user. Prefer reading real data with the filesystem "
    "and git tools, then analyze it with the dev_ops tools rather than guessing. "
    "When you need a diff, use the git diff tools and pass the raw diff text into the dev_ops tools. "
    "Finish with a concise, well-formatted summary of what you found and did."
)

agent = create_react_agent(llm, tools)
print("Agent ready with model:", MODEL)

### A helper to run the agent and show its reasoning

`pretty_print()` displays every step — the tool calls Gemini chose, the tool results, and the
final answer — which makes the LLM-driven policy visible.

In [ ]:
def ask_agent(query, verbose=True):
    result = run(agent.ainvoke(
        {"messages": [SystemMessage(SYSTEM_PROMPT), HumanMessage(query)]},
        config={"recursion_limit": 40},
    ))
    msgs = result["messages"]
    if verbose:
        for m in msgs:
            m.pretty_print()
    # compact list of the tools the model actually used
    used = [tc["name"] for m in msgs if isinstance(m, AIMessage)
            for tc in (m.tool_calls or [])]
    print("\n" + "=" * 60)
    print("Tools the agent chose:", used or "(none)")
    print("=" * 60)
    return result

# quick sanity check that the custom server responds
_ = ask_agent("Use the ping tool from dev_ops and tell me if the custom server is alive.")

## 8 · Demo — composition across servers

Each query forces Gemini to combine tools from **different** servers. Watch the printed steps:
the model chooses the path itself.

### Demo 1 — filesystem ➜ custom analysis
*List files, read `app.py`, then run `code_metrics` on its contents.*

In [ ]:
_ = ask_agent(
    "List the files in the workspace, read app.py, and use the dev_ops code_metrics tool "
    "to report its total lines, function count, and number of TODOs."
)

### Demo 2 — git ➜ custom changelog + commit message
*Inspect the uncommitted diff, then generate a changelog and a suggested commit message.*

In [ ]:
_ = ask_agent(
    "Show me the current git status and the unstaged diff. Then run generate_changelog and "
    "suggest_commit_message on that diff, and propose a commit message I could use."
)

### Demo 3 — full report across all three servers
*One prompt; Gemini orchestrates filesystem + git + dev_ops end to end.*

In [ ]:
_ = ask_agent(
    "Act as my dev assistant and give me a status report on this repository: "
    "(1) what files exist, "
    "(2) the last two commit messages from git log, "
    "(3) code_metrics for app.py, and "
    "(4) a suggested commit message for the current uncommitted changes. "
    "Present it as a short markdown summary."
)

## 9 · Wrap-up

**What this demonstrates**

- ✅ **Multiple MCP servers** orchestrated together — two third-party (`filesystem`, `git`) plus one **custom** (`dev_ops`).
- ✅ **stdio transport** — every server runs as a subprocess spawned by `MultiServerMCPClient`.
- ✅ **LLM-driven policy** — Gemini (`create_react_agent`) picks tools dynamically; there is no hard-coded flow.
- ✅ **Cross-server composition** — output of the git/filesystem tools flows into the custom analysis tools.

**Ideas to extend**
- Add a `run_tests` tool to `dev_ops` that shells out to `pytest` and summarizes failures.
- Add a **GitHub** MCP server to open issues / PRs from the agent.
- Give the agent memory (LangGraph checkpointer) so it can work across turns.
- Let it *write* a `CHANGELOG.md` back to disk via the filesystem server's write tool.